<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-axial-level-pairing/68_postE50_lumbar_morphometric_measurements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 68 · Post-E50 — Lumbar Morphometric Measurements

**Roadmap:** 67A → 67B/67B1/67B2 → 67C → **68 morphometric measurements**.

## Índice
- **68-A — Contrato, identidad de artefactos y scope**
- **68-B — Mediciones sagitales**
- **68-C — Mediciones axiales**
- **68-D — QC físico, estabilidad y auditoría**
- **68-E — Contrato final para Backend/Frontend**

### Contrato
- No entrena ni modifica modelos.
- Desarrollo técnico sobre **outer TRAIN** únicamente.
- `internal_test` y test oficial permanecen bloqueados.
- 67C permanece cerrado como **NOT VALIDATED**.
- Las mediciones son geométricas/revisables, no diagnósticas.
- Las clases axiales se conservan como `raw_*`: el repositorio no tiene nombres clínicos corroborados para ellas.
- **Runtime recomendado: NVIDIA L4.** 68-A/D/E son CPU-friendly; 68-B/C usan inferencia y conviene L4.


# 68-A — Contrato, identidad de artefactos y scope

In [1]:

import hashlib, json, math, os, random, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Mapping
import numpy as np
import pandas as pd

SEED=2026
random.seed(SEED); np.random.seed(SEED)

try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

PFI_DRIVE_ROOT=Path(os.environ.get("PFI_DRIVE_ROOT","/content/drive/MyDrive/PFI_MVP" if IN_COLAB else "./PFI_MVP"))
RSNA_ROOT=PFI_DRIVE_ROOT/"data"/"RSNA_LUMBAR_DISC"
TRAIN_IMAGES_ROOT=RSNA_ROOT/"train_images"
RESULTS_DIR=PFI_DRIVE_ROOT/"results"/"post_e50"/"68"
METRICS_DIR=PFI_DRIVE_ROOT/"metrics"/"post_e50"/"68"
for p in (RESULTS_DIR,METRICS_DIR): p.mkdir(parents=True,exist_ok=True)

SAGITTAL_CHECKPOINT_PATH=PFI_DRIVE_ROOT/"models"/"final"/"sagittal_spider_multiclass_final_best_cf11dcc0.pt"
AXIAL_CHECKPOINT_PATH=PFI_DRIVE_ROOT/"models"/"final"/"axial_t2_alkafri_final_v2_candidate.pt"

EXPECTED_SAGITTAL_SHA256="cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944"
EXPECTED_AXIAL_SHA256 = ("a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739")
EXPECTED_CSV_SHA256={
 "train.csv":"f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528",
 "train_label_coordinates.csv":"416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635",
 "train_series_descriptions.csv":"bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33",
}

RSNA_INTERNAL_TEST_LOCKED=True
OFFICIAL_TEST_ACCESSED=False
TRAINING_PERFORMED=False
POST_VALIDATION_RETUNING_67C=False

# Primera corrida: 30. Si todo pasa, cambiar SOLO a None.
MAX_STUDIES_68=30
SAVE_MASKS=False

assert RSNA_INTERNAL_TEST_LOCKED and not OFFICIAL_TEST_ACCESSED and not TRAINING_PERFORMED and not POST_VALIDATION_RETUNING_67C
print("IN_COLAB:",IN_COLAB)
print("PFI_DRIVE_ROOT:",PFI_DRIVE_ROOT)
print("MAX_STUDIES_68:",MAX_STUDIES_68)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True
PFI_DRIVE_ROOT: /content/drive/MyDrive/PFI_MVP
MAX_STUDIES_68: 30


In [2]:

def _importable(name):
    try:
        m=__import__(name); return getattr(m,"__version__","unknown")
    except ImportError: return None

req={k:_importable(k) for k in ("torch","pydicom","scipy","PIL")}
pkgs=[]
if req["pydicom"] is None: pkgs.append("pydicom")
if req["scipy"] is None: pkgs.append("scipy")
if req["PIL"] is None: pkgs.append("Pillow")
if pkgs: subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*pkgs])

import torch, torch.nn as nn, pydicom, scipy
from PIL import Image
from scipy import ndimage

torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:",DEVICE)
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))
else: print("WARNING: para 68-B/68-C usar L4.")


DEVICE: cuda
GPU: NVIDIA L4


In [3]:

CANONICAL_LEVELS=["L1-L2","L2-L3","L3-L4","L4-L5","L5-S1"]
SAGITTAL_CLASS_NAMES={0:"background",1:"vertebra_group",2:"canal",3:"disc_group"}
AXIAL_CLASS_NAMES={0:"background_250",1:"raw_0",2:"raw_50",3:"raw_100",4:"raw_150",5:"raw_200"}

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

def opaque_id(x): return hashlib.sha256(str(x).encode()).hexdigest()[:12]

def build_split(ids,seed=2026):
    ordered=sorted(str(x) for x in ids); rng=np.random.default_rng(seed); sh=list(ordered); rng.shuffle(sh)
    n=len(sh); a=int(n*.70); b=int(n*.15)
    return {"train":sorted(sh[:a]),"validation":sorted(sh[a:a+b]),"internal_test":sorted(sh[a+b:])}

for fn,exp in EXPECTED_CSV_SHA256.items():
    p=RSNA_ROOT/fn; assert p.is_file(),p
    got=sha256_file(p); print(fn,got,"MATCH" if got==exp else "MISMATCH"); assert got==exp

series_df=pd.read_csv(RSNA_ROOT/"train_series_descriptions.csv",dtype={"study_id":"string","series_id":"string"})
outer_split=build_split(series_df.study_id.dropna().unique())
assert {k:len(v) for k,v in outer_split.items()}=={"train":1382,"validation":296,"internal_test":297}

TRAIN_STUDY_IDS=outer_split["train"]
RUN_STUDY_IDS=TRAIN_STUDY_IDS if MAX_STUDIES_68 is None else TRAIN_STUDY_IDS[:MAX_STUDIES_68]
print("Outer split:",{k:len(v) for k,v in outer_split.items()})
print("68 scope: OUTER TRAIN ONLY | run:",len(RUN_STUDY_IDS))


train.csv f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528 MATCH
train_label_coordinates.csv 416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635 MATCH
train_series_descriptions.csv bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33 MATCH
Outer split: {'train': 1382, 'validation': 296, 'internal_test': 297}
68 scope: OUTER TRAIN ONLY | run: 30


In [4]:
from pathlib import Path

search_roots = [
    PFI_DRIVE_ROOT / "models",
    PFI_DRIVE_ROOT / "results",
    PFI_DRIVE_ROOT / "checkpoints",
]

candidates = []

for root in search_roots:
    if not root.exists():
        continue

    print("\nSearching:", root)

    for pattern in (
        "*axial*.pt",
        "*axial*.pth",
        "*alkafri*.pt",
        "*alkafri*.pth",
        "*v2*.pt",
        "*v2*.pth",
    ):
        found = list(root.rglob(pattern))
        print(pattern, "->", len(found))
        candidates.extend(found)

# remove duplicates
candidates = sorted(set(candidates))

print("\nTOTAL CANDIDATES:", len(candidates))

for p in candidates:
    print(p)


Searching: /content/drive/MyDrive/PFI_MVP/models
*axial*.pt -> 5
*axial*.pth -> 0
*alkafri*.pt -> 3
*alkafri*.pth -> 0
*v2*.pt -> 1
*v2*.pth -> 0

Searching: /content/drive/MyDrive/PFI_MVP/results
*axial*.pt -> 0
*axial*.pth -> 0
*alkafri*.pt -> 0
*alkafri*.pth -> 0
*v2*.pt -> 0
*v2*.pth -> 0

TOTAL CANDIDATES: 5
/content/drive/MyDrive/PFI_MVP/models/E10_axial_t2_final_training_clean_best.pt
/content/drive/MyDrive/PFI_MVP/models/E9_axial_t2_final_labels_unet_best_state_dict.pt
/content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_best.pt
/content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_v2_candidate.pt
/content/drive/MyDrive/PFI_MVP/models/quickcheck_training/axial_t2_alkafri_quickcheck.pt


In [5]:
EXPECTED = EXPECTED_AXIAL_SHA256

print("Expected SHA:", EXPECTED)
print()

matches = []

for i, p in enumerate(candidates, start=1):
    try:
        got = sha256_file(p)

        print(
            f"[{i}/{len(candidates)}]",
            p.name,
            got
        )

        if got == EXPECTED:
            matches.append(p)

    except Exception as e:
        print("ERROR:", p, type(e).__name__, e)

print("\nMATCHES:")

for p in matches:
    print(p)

assert len(matches) >= 1, (
    "No axial checkpoint matched the expected SHA."
)

AXIAL_CHECKPOINT_PATH = matches[0]

print("\nUsing:")
print(AXIAL_CHECKPOINT_PATH)

Expected SHA: a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739

[1/5] E10_axial_t2_final_training_clean_best.pt a8b91f563e101c9c4a3cf2c0ec84af29cbcfabd76eb76b2d06dbd13f7d7c78d6
[2/5] E9_axial_t2_final_labels_unet_best_state_dict.pt 4bf5551560592b6738734fb4ee33e21375621a9d1db1f362bb7c281d9f1adf03
[3/5] axial_t2_alkafri_final_best.pt a8b91f563e101c9c4a3cf2c0ec84af29cbcfabd76eb76b2d06dbd13f7d7c78d6
[4/5] axial_t2_alkafri_final_v2_candidate.pt a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739
[5/5] axial_t2_alkafri_quickcheck.pt 556ee120f6828821f75cebe4cd35557a37147e73ebd85517df88bd0cbb79db3a

MATCHES:
/content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_v2_candidate.pt

Using:
/content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_v2_candidate.pt


## Runtime compartido — arquitecturas exactas + morfometría física

In [6]:

class SDC(nn.Module):
    def __init__(self,i,o):
        super().__init__(); self.block=nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(self,x): return self.block(x)

class SagUNet(nn.Module):
    def __init__(self,num_classes=4,base_channels=16):
        super().__init__(); b=base_channels
        self.enc1=SDC(1,b); self.pool1=nn.MaxPool2d(2); self.enc2=SDC(b,b*2); self.pool2=nn.MaxPool2d(2); self.enc3=SDC(b*2,b*4); self.pool3=nn.MaxPool2d(2)
        self.bottleneck=SDC(b*4,b*8); self.up3=nn.ConvTranspose2d(b*8,b*4,2,2); self.dec3=SDC(b*8,b*4)
        self.up2=nn.ConvTranspose2d(b*4,b*2,2,2); self.dec2=SDC(b*4,b*2); self.up1=nn.ConvTranspose2d(b*2,b,2,2); self.dec1=SDC(b*2,b); self.out_conv=nn.Conv2d(b,num_classes,1)
    def forward(self,x):
        e1=self.enc1(x); e2=self.enc2(self.pool1(e1)); e3=self.enc3(self.pool2(e2)); z=self.bottleneck(self.pool3(e3))
        d3=self.dec3(torch.cat([self.up3(z),e3],1)); d2=self.dec2(torch.cat([self.up2(d3),e2],1)); d1=self.dec1(torch.cat([self.up1(d2),e1],1)); return self.out_conv(d1)

class ADC(nn.Module):
    def __init__(self,i,o):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(self,x): return self.net(x)

class AxUNet(nn.Module):
    def __init__(self,num_classes=6,base_channels=16):
        super().__init__(); b=base_channels
        self.e1=ADC(1,b); self.e2=ADC(b,b*2); self.e3=ADC(b*2,b*4); self.pool=nn.MaxPool2d(2); self.mid=ADC(b*4,b*8)
        self.u3=nn.ConvTranspose2d(b*8,b*4,2,2); self.d3=ADC(b*8,b*4); self.u2=nn.ConvTranspose2d(b*4,b*2,2,2); self.d2=ADC(b*4,b*2); self.u1=nn.ConvTranspose2d(b*2,b,2,2); self.d1=ADC(b*2,b); self.out=nn.Conv2d(b,num_classes,1)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); z=self.mid(self.pool(e3))
        x=self.d3(torch.cat([self.u3(z),e3],1)); x=self.d2(torch.cat([self.u2(x),e2],1)); x=self.d1(torch.cat([self.u1(x),e1],1)); return self.out(x)

def _state(ck):
    if isinstance(ck,Mapping):
        for k in ("model_state_dict","state_dict","model"):
            if isinstance(ck.get(k),Mapping): sd=ck[k]; break
        else: sd=ck
    else: raise ValueError("checkpoint")
    out={}
    for k,v in sd.items():
        kk=str(k)
        for p in ("module.","model."):
            if kk.startswith(p): kk=kk[len(p):]
        out[kk]=v
    return out

def load_model(path,kind):
    ck=torch.load(path,map_location=DEVICE,weights_only=False); sd=_state(ck)
    if kind=="sag":
        b=int(ck.get("base_channels",sd["enc1.block.0.weight"].shape[0])); n=int(ck.get("num_classes",sd["out_conv.weight"].shape[0])); m=SagUNet(n,b)
    else:
        b=int(ck.get("base_channels",sd["e1.net.0.weight"].shape[0])); n=int(ck.get("num_classes",sd["out.weight"].shape[0])); m=AxUNet(n,b)
    m.load_state_dict(sd,strict=True); m.to(DEVICE).eval()
    target=tuple(int(x) for x in ck.get("target_size",(256,256)))
    return m,target

sag_model,SAG_TARGET=load_model(SAGITTAL_CHECKPOINT_PATH,"sag")
ax_model,AX_TARGET=load_model(AXIAL_CHECKPOINT_PATH,"ax")
print("Models loaded:",SAG_TARGET,AX_TARGET)


Models loaded: (256, 256) (256, 256)


In [7]:

def robust_norm(a):
    x=np.asarray(a,dtype=np.float32); finite=np.isfinite(x)
    if not finite.any(): return np.zeros_like(x)
    lo,hi=np.percentile(x[finite],[1,99])
    if hi<=lo: return np.zeros_like(x)
    return np.clip((x-lo)/(hi-lo+1e-8),0,1).astype(np.float32)

def infer_mask(model,arr,target):
    x=robust_norm(arr); im=Image.fromarray((x*255).astype(np.uint8)).resize((target[1],target[0]),Image.Resampling.BILINEAR)
    t=torch.from_numpy(np.asarray(im,dtype=np.float32)[None,None]/255.0).to(DEVICE)
    with torch.inference_mode(): p=model(t).argmax(1)[0].cpu().numpy().astype(np.uint8)
    native=Image.fromarray(p).resize((arr.shape[1],arr.shape[0]),Image.Resampling.NEAREST)
    return np.asarray(native,dtype=np.uint8)

def read_dicom(p):
    ds=pydicom.dcmread(str(p)); arr=ds.pixel_array.astype(np.float32)
    spacing=tuple(float(v) for v in ds.PixelSpacing) if hasattr(ds,"PixelSpacing") else None
    return ds,arr,spacing

def ordered_paths(study,series):
    root=TRAIN_IMAGES_ROOT/str(study)/str(series); rec=[]
    for p in root.glob("*.dcm"):
        try:
            ds=pydicom.dcmread(str(p),stop_before_pixels=True,specific_tags=["InstanceNumber"])
            rec.append((int(getattr(ds,"InstanceNumber",0)),p))
        except Exception: pass
    return [p for _,p in sorted(rec)]

def comps(binary,min_pixels=20):
    lab,n=ndimage.label(binary.astype(np.uint8)); out=[]
    for i in range(1,n+1):
        m=lab==i
        if int(m.sum())>=min_pixels: out.append(m)
    return sorted(out,key=lambda m:int(m.sum()),reverse=True)

def morph(mask,spacing):
    if spacing is None or not mask.any(): return None
    rs,cs=map(float,spacing); rr,cc=np.where(mask); pts=np.column_stack([rr*rs,cc*cs]).astype(float); pts-=pts.mean(0)
    eig=np.linalg.eigh(np.cov(pts,rowvar=False))[1] if len(pts)>=3 else np.eye(2); pr=pts@eig
    lengths=np.ptp(pr,axis=0) if len(pts)>=3 else np.array([0.,0.]); major,minor=sorted(map(float,lengths),reverse=True)
    return {
      "area_mm2":float(mask.sum())*rs*cs,
      "bbox_height_mm":float(rr.max()-rr.min()+1)*rs,
      "bbox_width_mm":float(cc.max()-cc.min()+1)*cs,
      "major_axis_mm":major,"minor_axis_mm":minor,
      "centroid_row_px":float(rr.mean()),"centroid_col_px":float(cc.mean()),"pixel_count":int(mask.sum())
    }

m=np.zeros((10,20),bool); m[2:7,5:15]=True; q=morph(m,(2.,.5))
assert abs(q["area_mm2"]-50)<1e-9 and abs(q["bbox_height_mm"]-10)<1e-9 and abs(q["bbox_width_mm"]-5)<1e-9
print("Morphometry physical-unit self-test: PASS")


Morphometry physical-unit self-test: PASS


# 68-B — Mediciones sagitales

In [8]:

assert torch.cuda.is_available(),"68-B: usar GPU/L4."

rows=[]; failures=[]
sdf=series_df[series_df.study_id.astype(str).isin(set(map(str,RUN_STUDY_IDS))) & (series_df.series_description.astype(str).str.strip()=="Sagittal T2/STIR")]
t0=time.time()

for i,sid in enumerate(RUN_STUDY_IDS,1):
    s=sdf[sdf.study_id.astype(str)==str(sid)]
    if s.empty: failures.append({"study_id_opaque":opaque_id(sid),"reason":"NO_SAGITTAL_T2"}); continue
    ser=str(s.iloc[0].series_id); ps=ordered_paths(sid,ser)
    if not ps: failures.append({"study_id_opaque":opaque_id(sid),"reason":"NO_DICOM"}); continue
    p=ps[len(ps)//2]
    try:
        ds,arr,spacing=read_dicom(p); assert spacing is not None
        pred=infer_mask(sag_model,arr,SAG_TARGET)
        for cls,name,minpix in [(3,"disc_group_component",20),(2,"canal",20),(1,"vertebra_group_component",50)]:
            cc=comps(pred==cls,minpix)
            if cls==2: cc=cc[:1]
            else: cc=sorted(cc,key=lambda m:float(np.where(m)[0].mean()))
            for rank,c in enumerate(cc,1):
                mm=morph(c,spacing)
                rows.append({"study_id_opaque":opaque_id(sid),"series_id_opaque":opaque_id(ser),"instance_number":int(getattr(ds,"InstanceNumber",0)),"structure":name,"component_rank_superior_to_inferior_proxy":rank if cls!=2 else None,**mm})
    except Exception as e:
        failures.append({"study_id_opaque":opaque_id(sid),"reason":f"{type(e).__name__}:{e}"})
    if i%10==0 or i==len(RUN_STUDY_IDS): print(f"68-B {i}/{len(RUN_STUDY_IDS)} | {(time.time()-t0)/60:.1f} min")

sag_df=pd.DataFrame(rows); sag_fail_df=pd.DataFrame(failures)
SAG_PATH=RESULTS_DIR/"68_B_sagittal_measurements.csv"; SAG_FAIL_PATH=RESULTS_DIR/"68_B_sagittal_failures.csv"
sag_df.to_csv(SAG_PATH,index=False); sag_fail_df.to_csv(SAG_FAIL_PATH,index=False)
print("="*60); print("68-B SAGITTAL MEASUREMENTS COMPLETE"); print("="*60)
print("Rows:",len(sag_df)); print("Studies:",sag_df.study_id_opaque.nunique() if len(sag_df) else 0); print("Failures:",len(sag_fail_df))
print(sag_df.structure.value_counts().to_string() if len(sag_df) else "NONE")
print("SHA256:",sha256_file(SAG_PATH))


68-B 10/30 | 2.1 min
68-B 20/30 | 4.5 min
68-B 30/30 | 6.5 min
68-B SAGITTAL MEASUREMENTS COMPLETE
Rows: 670
Studies: 30
Failures: 0
structure
vertebra_group_component    418
disc_group_component        222
canal                        30
SHA256: d9174b458bd5d98971ede4344f57a5034a4ddc97bf31f8f758b70ea7dbb45a0b


# 68-C — Mediciones axiales

In [9]:

assert torch.cuda.is_available(),"68-C: usar GPU/L4."

rows=[]; failures=[]
adf=series_df[series_df.study_id.astype(str).isin(set(map(str,RUN_STUDY_IDS))) & series_df.series_description.astype(str).str.upper().str.contains("AXIAL",na=False)]
t0=time.time()

for i,sid in enumerate(RUN_STUDY_IDS,1):
    ss=adf[adf.study_id.astype(str)==str(sid)]
    if ss.empty: failures.append({"study_id_opaque":opaque_id(sid),"reason":"NO_AXIAL_SERIES"}); continue
    for _,sr in ss.iterrows():
        ser=str(sr.series_id); ps=ordered_paths(sid,ser)
        if not ps: failures.append({"study_id_opaque":opaque_id(sid),"series_id_opaque":opaque_id(ser),"reason":"NO_DICOM"}); continue
        p=ps[len(ps)//2]
        try:
            ds,arr,spacing=read_dicom(p); assert spacing is not None
            pred=infer_mask(ax_model,arr,AX_TARGET)
            for cls in range(1,6):
                name=AXIAL_CLASS_NAMES[cls]; union=pred==cls
                cc=comps(union,10)
                if cc:
                    rows.append({"study_id_opaque":opaque_id(sid),"series_id_opaque":opaque_id(ser),"instance_number":int(getattr(ds,"InstanceNumber",0)),"class_id":cls,"class_name":name,"measurement_scope":"largest_component","anatomical_name_confirmed":False,"axial_level_pairing_validated":False,**morph(cc[0],spacing)})
                if union.any():
                    rows.append({"study_id_opaque":opaque_id(sid),"series_id_opaque":opaque_id(ser),"instance_number":int(getattr(ds,"InstanceNumber",0)),"class_id":cls,"class_name":name,"measurement_scope":"class_union","anatomical_name_confirmed":False,"axial_level_pairing_validated":False,**morph(union,spacing)})
        except Exception as e:
            failures.append({"study_id_opaque":opaque_id(sid),"series_id_opaque":opaque_id(ser),"reason":f"{type(e).__name__}:{e}"})
    if i%10==0 or i==len(RUN_STUDY_IDS): print(f"68-C {i}/{len(RUN_STUDY_IDS)} | {(time.time()-t0)/60:.1f} min")

ax_df=pd.DataFrame(rows); ax_fail_df=pd.DataFrame(failures)
AX_PATH=RESULTS_DIR/"68_C_axial_rawclass_measurements.csv"; AX_FAIL_PATH=RESULTS_DIR/"68_C_axial_failures.csv"
ax_df.to_csv(AX_PATH,index=False); ax_fail_df.to_csv(AX_FAIL_PATH,index=False)
print("="*60); print("68-C AXIAL RAW-CLASS MEASUREMENTS COMPLETE"); print("="*60)
print("Rows:",len(ax_df)); print("Studies:",ax_df.study_id_opaque.nunique() if len(ax_df) else 0); print("Failures:",len(ax_fail_df))
print(ax_df.class_name.value_counts().to_string() if len(ax_df) else "NONE")
print("SHA256:",sha256_file(AX_PATH))
print("Anatomical names confirmed: False")
print("67C pairing formally validated: False")


68-C 10/30 | 3.7 min
68-C 20/30 | 8.3 min
68-C 30/30 | 12.3 min
68-C AXIAL RAW-CLASS MEASUREMENTS COMPLETE
Rows: 342
Studies: 30
Failures: 0
class_name
raw_0      74
raw_100    70
raw_50     66
raw_150    66
raw_200    66
SHA256: 741b43f61e250c2f27789bc6047f3028c2396f1b1bd7951441e396b94b4ddd21
Anatomical names confirmed: False
67C pairing formally validated: False


# 68-D — QC físico, estabilidad y auditoría

In [10]:

def audit(df,name):
    if df.empty: return {"name":name,"rows":0,"status":"NO_ROWS"}
    cols=["area_mm2","bbox_height_mm","bbox_width_mm","major_axis_mm","minor_axis_mm"]
    n=df[cols].apply(pd.to_numeric,errors="coerce")
    finite=np.isfinite(n.to_numpy()).all(1)
    pos=(n.area_mm2>0)&(n.bbox_height_mm>0)&(n.bbox_width_mm>0)
    axis=n.major_axis_mm+1e-9>=n.minor_axis_mm
    return {"name":name,"rows":int(len(df)),"finite_rate":float(finite.mean()),"positive_rate":float(pos.mean()),"major_ge_minor_rate":float(axis.mean()),"status":"PASS" if finite.all() and pos.all() and axis.all() else "FAIL"}

sag_qc=audit(sag_df,"sagittal"); ax_qc=audit(ax_df,"axial_raw_classes")

for d in (sag_df,ax_df):
    if not d.empty:
        assert "study_id" not in d.columns and "series_id" not in d.columns
        assert "study_id_opaque" in d.columns and "series_id_opaque" in d.columns

# Invariance simple: same physical pixel area product.
mask=np.ones((4,4),bool)
assert abs(morph(mask,(1,1))["area_mm2"]-16)<1e-9
assert abs(morph(mask,(2,.5))["area_mm2"]-16)<1e-9

qc={"stage":"68-D","generated_at":datetime.now(timezone.utc).isoformat(),"run_studies":len(RUN_STUDY_IDS),"scope":"OUTER_TRAIN_ONLY","sagittal_qc":sag_qc,"axial_qc":ax_qc,"pixel_spacing_invariance":"PASS","internal_test_accessed":False,"official_test_accessed":False,"training_performed":False}
QC_PATH=METRICS_DIR/"68_D_qc_summary.json"; QC_PATH.write_text(json.dumps(qc,indent=2))
print("="*60); print("68-D TECHNICAL QC"); print("="*60)
print("Sagittal:",sag_qc); print("Axial:",ax_qc); print("PixelSpacing invariance: PASS")
print("SHA256:",sha256_file(QC_PATH))


68-D TECHNICAL QC
Sagittal: {'name': 'sagittal', 'rows': 670, 'finite_rate': 1.0, 'positive_rate': 1.0, 'major_ge_minor_rate': 1.0, 'status': 'PASS'}
Axial: {'name': 'axial_raw_classes', 'rows': 342, 'finite_rate': 1.0, 'positive_rate': 1.0, 'major_ge_minor_rate': 1.0, 'status': 'PASS'}
PixelSpacing invariance: PASS
SHA256: 570592141664e2cc8d8663fd702a05a39007674aecd692202587a1971191d3e5


# 68-E — Contrato final para Backend/Frontend

In [12]:
# Recuperar identidades de checkpoints para 68-E

assert SAGITTAL_CHECKPOINT_PATH.is_file()
assert AXIAL_CHECKPOINT_PATH.is_file()

sag_sha = sha256_file(SAGITTAL_CHECKPOINT_PATH)
ax_sha = sha256_file(AXIAL_CHECKPOINT_PATH)

assert sag_sha == EXPECTED_SAGITTAL_SHA256
assert ax_sha == EXPECTED_AXIAL_SHA256

print("Sagittal SHA:", sag_sha)
print("Axial SHA:", ax_sha)
print("Checkpoint identities restored: PASS")

Sagittal SHA: cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944
Axial SHA: a48cbddd858b5615010fd809412f3d17dae6871fbe12a38f4720e6f6bc70f739
Checkpoint identities restored: PASS


In [13]:

contract={
 "contract_version":"68.1",
 "purpose":"NON_DIAGNOSTIC_REVIEWABLE_LUMBAR_MORPHOMETRY",
 "units":{"linear":"mm","area":"mm2"},
 "sagittal":{
   "source_model":"sagittal_spider","checkpoint_sha256":sag_sha,"semantic_classes":SAGITTAL_CLASS_NAMES,
   "measurements":["area_mm2","bbox_height_mm","bbox_width_mm","major_axis_mm","minor_axis_mm"],
   "absolute_level_source_runtime":"67B2","absolute_level_status":"VALIDATED_WITH_ABSTENTION","human_review_required":True
 },
 "axial":{
   "source_model":"axial_t2_alkafri_final_v2_candidate","checkpoint_sha256":ax_sha,"semantic_classes":AXIAL_CLASS_NAMES,
   "measurements":["area_mm2","bbox_height_mm","bbox_width_mm","major_axis_mm","minor_axis_mm"],
   "clinical_anatomical_names_confirmed":False,"level_pairing_source_runtime":"67C",
   "level_pairing_status":"EXPERIMENTAL_REVIEW_REQUIRED_NOT_FORMALLY_VALIDATED","human_review_required":True
 },
 "prohibited_interpretation":[
   "No diagnosis is produced.",
   "No stenosis grade is inferred from a morphometric threshold.",
   "No pathology is inferred from area or diameter alone.",
   "Axial raw classes must not be renamed clinically without documentary corroboration."
 ],
 "scope":{"development_data":"OUTER_TRAIN_ONLY","internal_test_accessed":False,"official_test_accessed":False,"training_performed":False}
}
CONTRACT_PATH=METRICS_DIR/"68_E_measurement_contract.json"; CONTRACT_PATH.write_text(json.dumps(contract,indent=2))

api_example={"studyId":"<runtime-study-id>","plane":"sagittal|axial","level":"L1-L2|L2-L3|L3-L4|L4-L5|L5-S1|null","levelStatus":"validated|abstained|experimental_review_required","structure":"disc_group_component|canal|vertebra_group_component|raw_*","measurements":{"areaMm2":None,"bboxHeightMm":None,"bboxWidthMm":None,"majorAxisMm":None,"minorAxisMm":None},"reviewRequired":True,"diagnostic":False}
API_PATH=METRICS_DIR/"68_E_api_example.json"; API_PATH.write_text(json.dumps(api_example,indent=2))

summary={"stage":"68-E","run_studies":len(RUN_STUDY_IDS),"sagittal_rows":int(len(sag_df)),"axial_rows":int(len(ax_df)),"sagittal_failures":int(len(sag_fail_df)),"axial_failures":int(len(ax_fail_df)),"sagittal_qc":sag_qc["status"],"axial_qc":ax_qc["status"],"contract_sha256":sha256_file(CONTRACT_PATH),"api_sha256":sha256_file(API_PATH),"internal_test_accessed":False,"official_test_accessed":False,"training_performed":False,"ready_for_backend_integration":bool(sag_qc["status"]=="PASS" and ax_qc["status"] in ("PASS","NO_ROWS"))}
SUM_PATH=METRICS_DIR/"68_E_final_summary.json"; SUM_PATH.write_text(json.dumps(summary,indent=2))

print("="*68); print("68-E MORPHOMETRIC MEASUREMENT CONTRACT COMPLETE"); print("="*68)
print("Studies:",len(RUN_STUDY_IDS)); print("Sag rows:",len(sag_df),"failures:",len(sag_fail_df)); print("Ax rows:",len(ax_df),"failures:",len(ax_fail_df))
print("Sag QC:",sag_qc["status"]); print("Ax QC:",ax_qc["status"])
print("Axial anatomical names confirmed: False"); print("67C pairing formally validated: False")
print("Contract SHA256:",sha256_file(CONTRACT_PATH)); print("API SHA256:",sha256_file(API_PATH)); print("Final summary SHA256:",sha256_file(SUM_PATH))
print("READY FOR BACKEND INTEGRATION:",summary["ready_for_backend_integration"])


68-E MORPHOMETRIC MEASUREMENT CONTRACT COMPLETE
Studies: 30
Sag rows: 670 failures: 0
Ax rows: 342 failures: 0
Sag QC: PASS
Ax QC: PASS
Axial anatomical names confirmed: False
67C pairing formally validated: False
Contract SHA256: 6d496e283074d00acaac601163e988b50bd1ce54f83272cc536416d5e6f52385
API SHA256: 07fa34833623ca75e2173df73a280a34f94a2fb46e0fd9c1e4b8766250211a01
Final summary SHA256: 2580657940beeadf9ab5b04bb6b57a98d70ea586939d3e2dc267d8f06d8d1167
READY FOR BACKEND INTEGRATION: True



# Ejecución

Primera corrida:
1. Colab **NVIDIA L4**.
2. Dejar `MAX_STUDIES_68 = 30`.
3. **Run all**.
4. Si aparece `AssertionError`, SHA mismatch, error de checkpoint, o QC `FAIL`, detenerse y pasar el traceback.
5. Si 68-A/B/C/D/E terminan bien, cambiar **solamente** `MAX_STUDIES_68 = None` y volver a ejecutar para outer TRAIN completo.

Pasar al chat los bloques finales de 68-A, 68-B, 68-C, 68-D y 68-E.
